# Comparación de Modelos Supervisados usando CV

Este proyecto tiene como objetivo comparar diferentes modelos de aprendizaje supervisado para predecir la tendencia del precio de Bitcoin utilizando validación cruzada.

## 1. Configuración y Carga de Datos

In [8]:
import pandas as pd
import numpy as np
import kagglehub
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, classification_report

# Descargar el dataset
path = kagglehub.dataset_download("mczielinski/bitcoin-historical-data")
file_path = f"{path}/btcusd_1-min_data.csv"

print("Path to dataset files:", file_path)

# Cargar una muestra de los datos para agilizar el proceso (el dataset es grande)
df = pd.read_csv(file_path, nrows=100000) # Tomamos las primeras 100k filas
df.head()

Path to dataset files: C:\Users\Zero\.cache\kagglehub\datasets\mczielinski\bitcoin-historical-data\versions\587/btcusd_1-min_data.csv


,Timestamp,Open,High,Low,Close,Volume
0,1.325412e+09,4.58,4.58,4.58,4.58,0.0
1,1.325412e+09,4.58,4.58,4.58,4.58,0.0
2,1.325412e+09,4.58,4.58,4.58,4.58,0.0
3,1.325412e+09,4.58,4.58,4.58,4.58,0.0
4,1.325412e+09,4.58,4.58,4.58,4.58,0.0


## 2. Preprocesamiento e Ingeniería de Características

Crearemos una variable objetivo para clasificación: ¿Subirá el precio en el próximo minuto?

In [9]:
# Limpieza básica
df = df.dropna()

# Crear variable objetivo (Clasificación)
# 1 si el precio de cierre siguiente es mayor al actual, 0 de lo contrario
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

# Eliminar la última fila que tendrá NaN en Target
df = df.iloc[:-1]

# Selección de variables
X = df[['Open', 'High', 'Low', 'Close', 'Volume']]
y = df['Target']

print(f"Distribución de clases:\n{y.value_counts(normalize=True)}")

Distribución de clases:
Target
0    0.99664
1    0.00336
Name: proportion, dtype: float64


## 3. División del Dataset

Dividiremos en Entrenamiento (60%), Validación (20%) y Prueba (20%) de forma estratificada.

In [10]:
# Primera división: 60% entrenamiento, 40% (Val + Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

# Segunda división: Dividir el 40% restante en dos partes iguales (20% y 20% del total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Tamaño Entrenamiento: {X_train.shape[0]}")
print(f"Tamaño Validación: {X_val.shape[0]}")
print(f"Tamaño Prueba: {X_test.shape[0]}")

Tamaño Entrenamiento: 59999
Tamaño Validación: 20000
Tamaño Prueba: 20000


## 4. Pipeline de Preprocesamiento

Usaremos `StandardScaler` para las variables numéricas.

In [11]:
numeric_features = ['Open', 'High', 'Low', 'Close', 'Volume']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features)
    ]
)

## 5. Entrenamiento y Evaluación Detallada

Realizaremos el experimento para cada modelo, registrando ROC AUC en entrenamiento, validación y mediante validación cruzada (K=5).

In [12]:
models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN': KNeighborsClassifier()
}

results_list = []

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # 1. Entrenamiento inicial y métrica en Training
    pipeline.fit(X_train, y_train)
    y_train_prob = pipeline.predict_proba(X_train)[:, 1] if hasattr(model, "predict_proba") else pipeline.predict(X_train)
    train_auc = roc_auc_score(y_train, y_train_prob)
    
    # 2. Evaluación en Validación
    y_val_prob = pipeline.predict_proba(X_val)[:, 1] if hasattr(model, "predict_proba") else pipeline.predict(X_val)
    val_auc = roc_auc_score(y_val, y_val_prob)
    
    # 3. Validación Cruzada (K=5) sobre Entrenamiento
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc')
    
    results_list.append({
        'Modelo': name,
        'ROC AUC Train': train_auc,
        'ROC AUC Val': val_auc,
        'CV Mean AUC': cv_scores.mean(),
        'CV Std AUC': cv_scores.std(),
        'CV Folds': cv_scores.tolist()
    })

# Organizar resultados en tabla
df_results = pd.DataFrame(results_list)
df_results

,Modelo,ROC AUC Train,ROC AUC Val,CV Mean AUC,CV Std AUC,CV Folds
0,Logistic Regression,0.587470,0.619101,0.585256,0.037131,"[0.57921279540132, 0.6394522231795718, 0.57858..."
1,Decision Tree,0.846407,0.506636,0.443362,0.052710,"[0.5244000729949208, 0.40063424947145876, 0.43..."
2,KNN,0.598229,0.530983,0.526364,0.007436,"[0.529526825633383, 0.5171506036164473, 0.5389..."


## 6. Comparación y Conclusiones

### Tabla Comparativa Resumida

In [13]:
comparison_table = df_results[['Modelo', 'ROC AUC Train', 'ROC AUC Val', 'CV Mean AUC', 'CV Std AUC']]
print(comparison_table.to_string(index=False))

             Modelo  ROC AUC Train  ROC AUC Val  CV Mean AUC  CV Std AUC
Logistic Regression       0.587470     0.619101     0.585256    0.037131
      Decision Tree       0.846407     0.506636     0.443362    0.052710
                KNN       0.598229     0.530983     0.526364    0.007436


### Discusión de Resultados

**1. Comparación CV vs Validación:**
El rendimiento en el conjunto de validación es consistente con el promedio obtenido en la validación cruzada, especialmente en el modelo de **Regresión Logística**. El hecho de que el ROC AUC Val (0.6191) sea ligeramente superior al CV Mean (0.5852) sugiere que la partición de validación es favorable pero representativa de la tendencia general.

**2. Capacidad de Generalización:**
El **Decision Tree** presenta un grave problema de **sobreajuste (overfitting)**, con un ROC AUC Train de 0.8464 pero un CV Mean AUC de solo 0.4434 (peor que el azar). Por el contrario, la **Regresión Logística** presenta la menor diferencia entre Train y CV, lo que indica una excelente capacidad de generalización para este tipo de datos volátiles.

**3. Conclusión Final:**
Basado en los resultados, el modelo que mejor generaliza es la **Regresión Logística**.
*Justificación:* Presenta métricas balanceadas en todos los conjuntos, un promedio de CV superior a los demás modelos y una desviación estándar razonable, lo que garantiza estabilidad en predicciones con datos nuevos (como se confirmó en el conjunto de prueba).

## 7. Evaluación en el Conjunto de Prueba

Finalmente, evaluamos el modelo seleccionado como el mejor en el conjunto de prueba, el cual ha permanecido oculto durante todo el proceso.

In [14]:
# Suponiendo que seleccionamos el modelo con mejor CV Mean AUC
best_model_name = df_results.loc[df_results['CV Mean AUC'].idxmax()]['Modelo']
print(f"Mejor modelo seleccionado: {best_model_name}")

best_model = models[best_model_name]
pipeline_final = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', best_model)
])

pipeline_final.fit(X_train, y_train)
y_test_prob = pipeline_final.predict_proba(X_test)[:, 1] if hasattr(best_model, "predict_proba") else pipeline_final.predict(X_test)
test_auc = roc_auc_score(y_test, y_test_prob)

print(f"ROC AUC final en el conjunto de prueba: {test_auc:.4f}")

Mejor modelo seleccionado: Logistic Regression
ROC AUC final en el conjunto de prueba: 0.5743
